# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/SanniyaAslam/flyrank-ml-internship/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [2]:
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_DIR = "/content/flyrank-ml-internship"

if IN_COLAB and not os.path.isdir(REPO_DIR):
    subprocess.run(["git", "clone", "--depth", "1",
                     "https://github.com/SanniyaAslam/flyrank-ml-internship.git", REPO_DIR], check=True)
os.chdir(REPO_DIR)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "duckdb", "huggingface_hub", "scikit-learn"], check=True)

from google.colab import userdata
HF_TOKEN = userdata.get("HF_TOKEN")

import duckdb, pandas as pd, numpy as np
con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")
REL = "hf://datasets/FlyRank/internship-warehouse"
print("Connected. Working dir:", os.getcwd())

Connected. Working dir: /content/flyrank-ml-internship


## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*I'm using Logistic Regression as my primary model, with a Random Forest as
a comparison. Logistic Regression fits this lane because the task is
binary classification (declining vs not) on tabular, mostly-linear
signals (impressions, CTR, position, staleness) — and because it's
directly readable: the coefficients tell a content reviewer exactly which
signals push a page toward "declining," which matters for a
decision-support tool a human has to trust. Random Forest is included as
a check for whether nonlinear interactions between signals add real
value, or whether the extra complexity isn't earning its keep — matching
the "don't reward complexity alone" rule for this assignment.

In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*I'm using a client-grouped split (GroupShuffleSplit on client_hash_id),
not a random row split. Rows for the same client are correlated — if one
page from a client lands in training and another in test, the model can
partly "memorize" that client's baseline behavior instead of learning
generalizable signal, which is the same leakage lesson from notebook 02
applied to a real split. I also build the label from the SECOND half of
March using only FIRST-half features, so the model never sees the future
window it's being scored against.

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
from sklearn.model_selection import GroupShuffleSplit
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score

df = con.sql(f"""
    WITH first_half AS (
        SELECT client_hash_id, content_hash_id,
               SUM(gsc_impressions) AS impressions_fh,
               SUM(gsc_clicks) AS clicks_fh,
               AVG(gsc_avg_position) AS avg_position_fh
        FROM read_parquet('{REL}/fact_content_daily_performance/month=2026-03/*.parquet')
        WHERE gsc_data_available IS TRUE AND EXTRACT(day FROM report_date) <= 15
        GROUP BY client_hash_id, content_hash_id
    ),
    second_half AS (
        SELECT client_hash_id, content_hash_id,
               SUM(gsc_clicks) AS clicks_sh
        FROM read_parquet('{REL}/fact_content_daily_performance/month=2026-03/*.parquet')
        WHERE gsc_data_available IS TRUE AND EXTRACT(day FROM report_date) > 15
        GROUP BY client_hash_id, content_hash_id
    )
    SELECT
        fh.client_hash_id, fh.content_hash_id,
        fh.impressions_fh, fh.clicks_fh, fh.avg_position_fh,
        sh.clicks_sh,
        d.content_updated_date, d.word_count,
        DATE_DIFF('day', d.content_updated_date, DATE '2026-03-15') AS days_since_update_fh
    FROM first_half fh
    JOIN second_half sh ON fh.client_hash_id = sh.client_hash_id AND fh.content_hash_id = sh.content_hash_id
    JOIN read_parquet('{REL}/dim_content.parquet') d
      ON fh.client_hash_id = d.client_hash_id AND fh.content_hash_id = d.content_hash_id
    WHERE fh.impressions_fh > 0
""").df()

df["ctr_fh"] = df["clicks_fh"] / df["impressions_fh"]
df["is_declining"] = (df["clicks_sh"] < df["clicks_fh"]).astype(int)
df["word_count"] = df["word_count"].fillna(df["word_count"].median())
df["days_since_update_fh"] = df["days_since_update_fh"].clip(lower=0)

print("Rows:", len(df), "| declining rate:", round(df["is_declining"].mean(), 3))

features = ["impressions_fh", "ctr_fh", "avg_position_fh", "days_since_update_fh", "word_count"]
X = df[features].replace([np.inf, -np.inf], np.nan).fillna(0)
y = df["is_declining"].values
groups = df["client_hash_id"].values

gss = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=42)
train_idx, test_idx = next(gss.split(X, y, groups))

train_clients = set(df.iloc[train_idx]["client_hash_id"])
test_clients = set(df.iloc[test_idx]["client_hash_id"])
print("Train rows:", len(train_idx), "| Test rows:", len(test_idx))
print("Client overlap between train/test (should be 0):", len(train_clients & test_clients))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Rows: 141467 | declining rate: 0.203
Train rows: 129917 | Test rows: 11550
Client overlap between train/test (should be 0): 0


## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*Metric: Precision@50, matching my Week-4 baseline, computed on the SAME
held-out test rows for every method. Baseline = the Week-4 rule
(days_since_update >= 180, impressions >= 100, ctr < 2%), recomputed here
using only first-half features so it's a fair apples-to-apples comparison
against the model, scored on the same test split.

In [5]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
def precision_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores))
    return np.asarray(labels)[order[:k]].mean()

X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
y_train, y_test = y[train_idx], y[test_idx]
df_test = df.iloc[test_idx].reset_index(drop=True)

# Baseline rule, recomputed on test rows only
baseline_flag = (
    (df_test["days_since_update_fh"] >= 180) &
    (df_test["impressions_fh"] >= 100) &
    (df_test["ctr_fh"] < 0.02)
).astype(int)
baseline_score = baseline_flag * df_test["impressions_fh"] * df_test["days_since_update_fh"]
baseline_p50 = precision_at_k(baseline_score, y_test, 50)

# Logistic Regression
logreg = LogisticRegression(max_iter=1000, class_weight="balanced")
logreg.fit(X_train, y_train)
logreg_scores = logreg.predict_proba(X_test)[:, 1]
logreg_p50 = precision_at_k(logreg_scores, y_test, 50)
logreg_auc = roc_auc_score(y_test, logreg_scores)

# Random Forest
rf = RandomForestClassifier(n_estimators=200, max_depth=6, class_weight="balanced", random_state=42)
rf.fit(X_train, y_train)
rf_scores = rf.predict_proba(X_test)[:, 1]
rf_p50 = precision_at_k(rf_scores, y_test, 50)
rf_auc = roc_auc_score(y_test, rf_scores)

results = pd.DataFrame({
    "method": ["Baseline rule (Week 4)", "Logistic Regression", "Random Forest"],
    "precision_at_50": [baseline_p50, logreg_p50, rf_p50],
    "roc_auc": [np.nan, logreg_auc, rf_auc]
})
print("=== Model vs Baseline ===")
display(results)

=== Model vs Baseline ===


,method,precision_at_50,roc_auc
0,Baseline rule (Week 4),0.20,NaN
1,Logistic Regression,0.78,0.874546
2,Random Forest,0.88,0.908646


Model vs Baseline: Random Forest (Precision@50 = 0.88, ROC AUC = 0.909)
and Logistic Regression (0.78, 0.875) both dramatically beat the Week-4
rule baseline (0.20) on the same client-grouped test split. The learned
models find far more true "declining" pages in their top 50 than the
hand-written stale+low-CTR rule did.

Where the model is wrong: Looking at the top missed declining pages
(false negatives), these are pages with real visibility (1,300-12,500
impressions) and near-zero CTR (0.04%-0.08%) that the model still scored
only moderately (~0.69-0.70) — likely because avg_position for several of
these is quite good (1.9-2.4), and the model may be under-weighting "high
impressions + near-zero CTR at a good position" as a strong decline
signal on its own.

The false alarms tell an opposite story: these are very low-impression
pages (5-16 impressions) that the model scored highly (0.90+) despite
tiny sample sizes — a CTR of, say, 20% on 5 impressions (1 click) is
statistically noisy, not a real signal, but the model appears to treat it
with the same confidence as a CTR measured over thousands of impressions.
This suggests the model is leaning on ctr_fh without accounting for how
unreliable that ratio is at very low impression counts — a good candidate
for adding a minimum-impressions filter or a smoothed/regularized CTR
feature in a future iteration.

Notably, days_since_update_fh = 0 for every row shown in both error
tables, echoing the staleness data-quality issue flagged in my Week-4
baseline: staleness isn't distinguishing these particular error cases,
reinforcing that avg_position and (unreliable, low-volume) ctr_fh are
doing most of the work here, not staleness.

In [6]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
coef_table = pd.DataFrame({
    "feature": features,
    "logreg_coefficient": logreg.coef_[0],
    "rf_importance": rf.feature_importances_
}).sort_values("rf_importance", ascending=False)
print("=== Feature importance / coefficients ===")
display(coef_table)

# Look at model's biggest mistakes
df_test["rf_score"] = rf_scores
false_negatives = df_test[(y_test == 1)].sort_values("rf_score").head(5)
false_positives = df_test[(y_test == 0)].sort_values("rf_score", ascending=False).head(5)

print("\n=== Top 5 missed declining pages (false negatives, low predicted score) ===")
display(false_negatives[["content_hash_id", "impressions_fh", "ctr_fh", "avg_position_fh", "days_since_update_fh", "rf_score"]])

print("\n=== Top 5 false alarms (predicted high, not actually declining) ===")
display(false_positives[["content_hash_id", "impressions_fh", "ctr_fh", "avg_position_fh", "days_since_update_fh", "rf_score"]])

=== Feature importance / coefficients ===


,feature,logreg_coefficient,rf_importance
1,ctr_fh,74.705229,0.820514
0,impressions_fh,0.000347,0.154026
2,avg_position_fh,-0.026202,0.019962
4,word_count,0.000181,0.004674
3,days_since_update_fh,-0.002652,0.000824



=== Top 5 missed declining pages (false negatives, low predicted score) ===


,content_hash_id,impressions_fh,ctr_fh,avg_position_fh,days_since_update_fh,rf_score
6721,content_8b898761780f350c,12567.0,0.000080,42.077033,0,0.691058
504,content_07e296eb7942b6dc,2504.0,0.000399,2.409042,0,0.696348
9347,content_59482bab891c1368,1333.0,0.000750,1.855321,0,0.698233
2467,content_667f5dfc446b591c,1327.0,0.000754,1.956019,0,0.698233
2590,content_85174c59769b20a9,2015.0,0.000496,2.963690,0,0.700789



=== Top 5 false alarms (predicted high, not actually declining) ===


,content_hash_id,impressions_fh,ctr_fh,avg_position_fh,days_since_update_fh,rf_score
4879,content_bfc3049a84cf287a,11.0,0.090909,15.600000,0,0.908561
3974,content_73c45865be5c672d,5.0,0.200000,27.000000,0,0.906001
3803,content_658a48362f36873e,14.0,0.071429,10.750000,0,0.904664
8724,content_280e4f71dd5efaf9,15.0,0.133333,3.659091,0,0.901725
5736,content_2cd25898f7cf2f5c,16.0,0.062500,8.750000,0,0.900576


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.